In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
os.chdir('/content/drive/MyDrive/Stanford_CS336/assignment_1')

#2.1 The Unicode Standard


(a) What Unicode character does chr(0) return?

Deliverable: A one-sentence response.

In [ ]:
chr(0)

'\x00'

(b) How does this characters string representation ```(__repr__())``` differ from its printed representation?

Deliverable: A one-sentence response.

In [ ]:
chr(0)

'\x00'

In [ ]:
chr(0).__repr__()

"'\\x00'"

(c) What happens when this character occurs in text? It may be helpful to play around with the following in your Python interpreter and see if it matches your expectations:

Deliverable: A one-sentence response.

In [ ]:
chr(0)

'\x00'

In [ ]:
print(chr(0))

 


In [ ]:
'this is a test' + chr(0) + 'string'

'this is a test\x00string'

In [ ]:
print('this is a test' + chr(0) + 'string')

this is a test string


# 2.2 Unicode Encodings

(a) What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than
UTF-16 or UTF-32? It may be helpful to compare the output of these encodings for various
input strings.

Deliverable: A one-to-two sentence response.

In [ ]:
## Answer: UTF-8 is relatively shorter since all characters (even the simple ones) will have 4 bytes in length (for UTF-32) in comparison to 1 byte (for UTF-8)

test = "hello你😀"
test_8 = test.encode("utf-8")
test_16 = test.encode("utf-16")
test_32 = test.encode("utf-32")

print(test_8) #'h e l l o \xe4\xbd\xa0 \xf0\x9f\x98\x80'
print(test_16)#'\xff\xfe h\x00 e\x00 l\x00 l\x00 o\x00 `O =\xd8\x00\xde'
print(test_32)#'\xff\xfe\x00\x00 h\x00\x00\x00 e\x00\x00\x00 l\x00\x00\x00 l\x00\x00\x00 o\x00\x00\x00 `O\x00\x00 \x00\xf6\x01\x00'

b'hello\xe4\xbd\xa0\xf0\x9f\x98\x80'
b'\xff\xfeh\x00e\x00l\x00l\x00o\x00`O=\xd8\x00\xde'
b'\xff\xfe\x00\x00h\x00\x00\x00e\x00\x00\x00l\x00\x00\x00l\x00\x00\x00o\x00\x00\x00`O\x00\x00\x00\xf6\x01\x00'


(b) Consider the following (incorrect) function, which is intended to decode a UTF-8 byte string into
a Unicode string. Why is this function incorrect? Provide an example of an input byte string
that yields incorrect results.

Deliverable: An example input byte string for which decode_utf8_bytes_to_str_wrong produces incorrect output, with a one-sentence explanation of why the function is incorrect.



```python
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

>>> decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))
'hello'
```



In [ ]:
## Answer: characters in bytestring may not always have 1 bytes (i.e. 你 = \xe4\xbd\xa0)

def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

# decode_utf8_bytes_to_str_wrong('😀'.encode('utf-8')) #returns error

(c) Give a two byte sequence that does not decode to any Unicode character(s).

Deliverable: An example, with a one-sentence explanation.

In [ ]:
# Answer: \xe4\xbd

# bytestring = b"\xe4\xbd"
# [bytes([b]).decode('utf-8') for b in bytestring]

# 2.3 Subword Tokenization

#2.4 BPE Tokenizer Training


bpe example

In [ ]:
from collections import Counter
import regex as re


def pre_tokenize(data, special_token = ['\n', '\t', '\r']):
    # init vocab
    vocab = [bytes([b]) for b in range(256)]

    # use re to pre-tokenize (can also just split by space)
    data = data.translate(str.maketrans({sp:" " for sp in special_token}))
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    pre_tokenized = re.findall(PAT, sample_data)
    pre_tokenized = [tuple(bytes([char]) for char in word.encode('utf-8')) for word in pre_tokenized]

    return pre_tokenized, vocab


def convert(tok):
    if isinstance(tok, bytes):
        return tok
    # tok is a tuple of sub-tokens – flatten each of them
    return b"".join(convert(t) for t in tok)


def merge(pre_tokenized, max_size):
    # create an initial frequency table
    letter_table = Counter(pre_tokenized)

    merge_size = max_size - 256
    for _ in range(merge_size):
        # build the pair-frequency table
        pairs_table = {}
        for word, freq in letter_table.items():
            for i in range(len(word) - 1):
                pair = word[i:i+2]
                pairs_table[pair] = pairs_table.get(pair, 0) + freq

        # choose the most frequent pair
        max_freq  = max(pairs_table.values())
        top_pairs = [k for k, v in pairs_table.items() if v == max_freq]
        top_p     = max(top_pairs)

        # create the new token
        new_token = convert(top_p[0]) + convert(top_p[1])
        vocab.append(new_token)

        # rewrite every word using the new symbol
        new_letter_table = {}
        for word, freq in letter_table.items():
            new_word = []
            i = 0
            while i < len(word):
                if i + 1 < len(word) and word[i:i+2] == top_p:
                    new_word.append(new_token)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_letter_table[tuple(new_word)] = freq

        letter_table = new_letter_table



sample_data = "low low low low low\nlower lower widest widest widest\nnewest newest newest newest newest newest"
pre_tokenized, vocab = pre_tokenize(sample_data)
merge(pre_tokenized, 262)
print(vocab)

[b'\x00', b'\x01', b'\x02', b'\x03', b'\x04', b'\x05', b'\x06', b'\x07', b'\x08', b'\t', b'\n', b'\x0b', b'\x0c', b'\r', b'\x0e', b'\x0f', b'\x10', b'\x11', b'\x12', b'\x13', b'\x14', b'\x15', b'\x16', b'\x17', b'\x18', b'\x19', b'\x1a', b'\x1b', b'\x1c', b'\x1d', b'\x1e', b'\x1f', b' ', b'!', b'"', b'#', b'$', b'%', b'&', b"'", b'(', b')', b'*', b'+', b',', b'-', b'.', b'/', b'0', b'1', b'2', b'3', b'4', b'5', b'6', b'7', b'8', b'9', b':', b';', b'<', b'=', b'>', b'?', b'@', b'A', b'B', b'C', b'D', b'E', b'F', b'G', b'H', b'I', b'J', b'K', b'L', b'M', b'N', b'O', b'P', b'Q', b'R', b'S', b'T', b'U', b'V', b'W', b'X', b'Y', b'Z', b'[', b'\\', b']', b'^', b'_', b'`', b'a', b'b', b'c', b'd', b'e', b'f', b'g', b'h', b'i', b'j', b'k', b'l', b'm', b'n', b'o', b'p', b'q', b'r', b's', b't', b'u', b'v', b'w', b'x', b'y', b'z', b'{', b'|', b'}', b'~', b'\x7f', b'\x80', b'\x81', b'\x82', b'\x83', b'\x84', b'\x85', b'\x86', b'\x87', b'\x88', b'\x89', b'\x8a', b'\x8b', b'\x8c', b'\x8d', b'\x8e', b'

# 2.5 Experimenting with BPE Tokenizer Training

BPE Tokenizer Training
Deliverable: Write a function that, given a path to an input text file, trains a (byte-level) BPE
tokenizer. Your BPE training function should handle (at least) the following input parameters:

```input_path: str``` Path to a text file with BPE tokenizer training data.

```vocab_size: int``` A positive integer that defines the maximum final vocabulary size (including the initial byte vocabulary, vocabulary items produced from merging, and any special tokens).

```special_tokens: list[str]``` A list of strings to add to the vocabulary. These special tokens do not otherwise affect BPE training.


Your BPE training function should return the resulting vocabulary and merges:

```vocab: dict[int, bytes]``` The tokenizer vocabulary, a mapping from int (token ID in the vocabu-lary) to bytes (token bytes).

```merges: list[tuple[bytes, bytes]]``` A list of BPE merges produced from training. Each list item is a tuple of bytes (<token1>, <token2>), representing that <token1> was merged with <token2>. The merges should be ordered by order of creation.


To test your BPE training function against our provided tests, you will first need to implement the test adapter at [adapters.run_train_bpe]. Then, run uv run pytest tests/test_train_bpe.py. Your implementation should be able to pass all tests. Optionally (this could be a large time-investment), you can implement the key parts of your training method using some systems language, for instance C++ (consider cppyy for this) or Rust (using PyO3). If you do this, be aware of which operations require copying vs reading directly from Python memory, and make sure to leave build instructions, or make sure it builds using only pyproject.toml. Also note that the GPT-2 regex is not well-supported in most regex engines and will be too slow in most that do. We have verified that Oniguruma is reasonably fast and supports negative lookahead, but the regex package in Python is, if anything, even faster.

In [ ]:
from collections import Counter
import regex as re
from tqdm import tqdm
from collections import defaultdict

def pre_tokenize(data, special_token = ['\n', '\t', '\r']):
    # init vocab
    vocab = {b: bytes([b]) for b in range(256)}

    # use re to pre-tokenize (can also just split by space)
    for sp in special_token:
        data = data.replace(sp, ' ')
    PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    pre_tokenized = re.finditer(PAT, data) # returns a list of words
    counter = defaultdict(int)

    for tok in pre_tokenized:
        counter[tuple([i.to_bytes(1, 'big') for i in tok.group().encode("utf-8")])] += 1

    return counter, vocab

def convert(tok):
    if isinstance(tok, bytes):
        return tok
    # tok is a tuple of sub-tokens – flatten each of them
    return b"".join(convert(t) for t in tok)


def merge(pre_tokenized, max_size, vocab):
    # create an initial frequency table
    letter_table = Counter(pre_tokenized)
    merges = []
    merge_size = max_size - 256
    token_id = 256


    with tqdm(total=merge_size, desc="BPE Merges", unit="merge") as pbar:
        for _ in range(merge_size):
            # build the pair-frequency table
            pairs_table = {}
            for word, freq in letter_table.items():
                # print(word, freq, type(word))
                for i in range(len(word) - 1):
                    pair = word[i:i+2]
                    pairs_table[pair] = pairs_table.get(pair, 0) + freq

            # choose the most frequent pair
            if not pairs_table:
                break
            max_freq  = max(pairs_table.values())
            top_pairs = [k for k, v in pairs_table.items() if v == max_freq]
            top_p     = max(top_pairs)

            # create the new token
            new_token = convert(top_p[0]) + convert(top_p[1])
            merges.append((convert(top_p[0]), convert(top_p[1])))
            vocab[token_id] = new_token
            token_id += 1

            # rewrite every word using the new symbol
            new_letter_table = {}
            for word, freq in letter_table.items():
                new_word = []
                i = 0
                while i < len(word):
                    if i + 1 < len(word) and word[i:i+2] == top_p:
                        new_word.append(new_token)
                        i += 2
                    else:
                        new_word.append(word[i])
                        i += 1
                new_letter_table[tuple(new_word)] = freq

            letter_table = new_letter_table

            pbar.update(1)

    return vocab, merges

def BPE(input_path: str, data:str, vocab_size: int, special_tokens: list[str]):
    if not data:
        with open(input_path, encoding="utf-8") as f:
            data = f.read()

    print(data[:100])
    pre_tokenized, vocab = pre_tokenize(data, special_tokens)
    print('finished pretokentizing')
    print(pre_tokenized[:100])
    print('finished pretokentizing')
    vocab, merges = merge(pre_tokenized, vocab_size - len(special_tokens), vocab)
    return vocab, merges

(a) Train a byte-level BPE tokenizer on the TinyStories dataset, using a maximum vocabulary size of 10,000. Make sure to add the TinyStories <|endoftext> special token to the vocabulary.

Serialize the resulting vocabulary and merges to disk for further inspection. How many hours and memory did training take? What is the longest token in the vocabulary? Does it make sense?

Resource requirements: ≤ 30 minutes (no GPUs), ≤ 30GB RAM
Hint You should be able to get under 2 minutes for BPE training using multiprocessing during pretokenization and the following two facts:

(a.1) The <|endoftext|> token delimits documents in the data files.

(a.2) The <|endoftext|> token is handled as a special case before the BPE merges are applied.

Deliverable: A one-to-two sentence response.

In [ ]:
import urllib.request

urllib.request.urlretrieve("https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-train.txt", "train.txt",)
urllib.request.urlretrieve("https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main/TinyStories-valid.txt", "valid.txt",)

('valid.txt', <http.client.HTTPMessage at 0x7c9f64514090>)

In [ ]:
new_vocab, new_merges = BPE("train.txt", "", 10000, ['\n', '\t', '\r', ])

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with


KeyboardInterrupt: 

In [ ]:
print(new_vocab)
print(new_merges)

(b) Profile your code. What part of the tokenizer training process takes the most time?

Deliverable: A one-to-two sentence response.

In [ ]:
#Answer: Pre-tokenization takes more time.

(a) Train a byte-level BPE tokenizer on the OpenWebText dataset, using a maximum vocabulary
size of 32,000. Serialize the resulting vocabulary and merges to disk for further inspection. What
is the longest token in the vocabulary? Does it make sense?
Resource requirements: ≤ 12 hours (no GPUs), ≤ 100GB RAM
Deliverable: A one-to-two sentence response.
(b) Compare and contrast the tokenizer that you get training on TinyStories versus OpenWebText.
Deliverable: A one-to-two sentence response.

In [ ]:
#SKIPPED DUE TO TIME CONSTRAINT (REVISIT)

# 2.6 BPE Tokenizer: Encoding and Decoding

In [ ]:
import regex as re
import json
from typing import Any, Iterable, Iterator
from pathlib import Path

class Tokenizer():
    def __init__(self,
                 vocab: dict[int, bytes],
                 merges: list[tuple[bytes, bytes]],
                 special_tokens: list[str] | None = None):

        self.vocab = vocab
        special_tokens = [] if not special_tokens else special_tokens
        special_tokens.sort(key=len, reverse=True)
        self.vocab_inverse = {v: k for k, v in self.vocab.items()}
        self.merges = merges
        self.special_tokens = special_tokens
        return

    @classmethod
    def from_files(cls,
                   vocab_filepath: str | Path,
                   merges_filepath: str | Path,
                   special_tokens_filepath: str | None = None):

        raw_vocab: dict[str, str] = json.loads(Path(vocab_filepath).read_text("utf-8"))
        vocab: dict[int, bytes] = {
            int(k): v.encode("utf-8") for k, v in raw_vocab.items()
        }

        raw_merges: list[list[str]] = json.loads(Path(merges_filepath).read_text("utf-8"))
        merges: list[tuple[bytes, bytes]] = [
            (a.encode("utf-8"), b.encode("utf-8")) for a, b in raw_merges
        ]

        special_tokens: list[str] | None = None
        if special_tokens_filepath is not None:
            special_tokens = json.loads(Path(special_tokens_filepath).read_text("utf-8"))

        return cls(vocab, merges, special_tokens)

    def to_files(self,
                 vocab_filepath: str | Path,
                 merges_filepath: str | Path,
                 special_tokens_filepath: str | None = None) -> None:

        vocab_json: dict[str, str] = {
            str(idx): token.decode("utf-8") for idx, token in self.vocab.items()
        }
        Path(vocab_filepath).write_text(
            json.dumps(vocab_json, ensure_ascii=False, indent=2), encoding="utf-8"
        )

        merges_json: list[list[str]] = [
            [a.decode("utf-8"), b.decode("utf-8")] for a, b in self.merges
        ]
        Path(merges_filepath).write_text(
            json.dumps(merges_json, ensure_ascii=False, indent=2), encoding="utf-8"
        )

        if special_tokens_filepath is not None:
            Path(special_tokens_filepath).write_text(
                json.dumps(self.special_tokens, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )

    def encode(self, text:str) -> list[int]:
        # create document boundary by special tokens
        if self.special_tokens:
            delims   = self.special_tokens
            pattern  = "(" + "|".join(map(re.escape, delims)) + ")"
            parts = re.split(pattern, text)
        else:
            parts = [text]

        # encode each document chunk into list of ints
        encoded = []
        for chunk in parts:
            if not chunk:
                continue

            if chunk in self.special_tokens:
                encoded.append(self.vocab_inverse[chunk.encode('utf-8')])
                # print(f'found special token {chunk}')
                continue

            PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
            pre_tokenized = re.finditer(PAT, chunk)
            for tok in pre_tokenized:
                word = [i.to_bytes(1, 'big') for i in tok.group().encode("utf-8")]
                # print(f'found word {word}')

                for l, r in self.merges:
                    i = 0
                    new_word = []
                    while i < len(word):
                        if i < len(word)-1 and l == word[i] and r == word[i+1]:
                            new_word.append(l+r)
                            i += 2
                        else:
                            new_word.append(word[i])
                            i += 1
                    word = new_word

                for w in word:
                    encoded.append(self.vocab_inverse[w])
        return encoded

    def encode_iterable(self, iterable: Iterable[str]) -> Iterator[int]:
        for s in iterable:
            for t in self.encode(s):
                yield t


    def decode(self, ids):
        decoded_bytes = b''
        for id in ids:
            decoded_bytes += self.vocab[id]
        decoded = decoded_bytes.decode('utf-8', errors = 'replace')
        return decoded



In [ ]:
# vocab = {0: b' ', 1: b'a', 2:b'c', 3: b'e', 4: b'h', 5: b't', 6: b'th', 7: b' c', 8: b' a', 9: b'the', 10: b' at'}

new = {6: b'th', 7: b' c', 8: b' a', 9: b'the', 10: b' at'}
vocab = {b: bytes([b]) for b in range(256)}
for v in new.values():
    vocab[len(vocab)] = v

merges = [(b't', b'h'), (b' ', b'c'), (b' ', b'a'), (b'th', b'e'), (b' a', b't')]

tokenizer = Tokenizer(vocab, merges, special_tokens=['<end of text>'])

encoded = tokenizer.encode('你好')
decoded = tokenizer.decode(encoded)


print(encoded)
print(decoded)


[228, 189, 160, 229, 165, 189]
你好


# 2.7 Experiments

In [ ]:
#SKIPPED